## Question 3: Are Women Catching Up in Wages Within Education Groups Over Time?

**Research Hypothesis:** Question 2 showed that, in 2024, women earn less than men at every level of education. The natural follow-up is whether the gap is at least narrowing year by year. Given that women now make up the majority of new higher-education graduates in Denmark, we expect the female-to-male wage ratio to drift upward across the 2004–2024 period — and we expect the strongest movement at the higher-education levels, where the female credential surplus is largest. If the ratio is flat or falling, that would suggest education alone does not translate into income convergence.

**Why we created this plot:** A single 2024 snapshot cannot tell us whether the gap is closing, widening, or stuck. We therefore plot the female/male wage ratio as a yearly time series for each of the six education levels on a shared axis. A line chart makes the direction and slope of each group easy to read, the parity reference line at 1.0 turns the chart into an unambiguous yes/no test for catch-up, and putting all education levels on the same panel lets us compare which credential tiers are converging faster than others.

In [ ]:
# Define wage_income here — also used by the ML section below
education_order = [
    "Primary School", "Gymnasium", "Vocational Education",
    "Short Higher Education", "Medium Higher Education", "Long Higher Education",
]

wage_income = Data[
    (Data["Income_Type"] == "4 Wages/salary")
    & (Data["Gender"].isin(["Male", "Female"]))
    & (Data["Unit"] == "Average for persons with income type (DKK)")
    & (Data["Education_Level"] != "Unknown")
].copy()

# Plot 3: Female/Male wage/salary ratio over time by education level (national average)
trend_df = wage_income[wage_income["Education_Level"].isin(education_order)].copy()

national = trend_df.groupby(["Year", "Education_Level", "Gender"], as_index=False)["Value"].mean()

national_wide = (
    national.pivot_table(
        index=["Year", "Education_Level"], columns="Gender", values="Value", aggfunc="mean"
    )
    .dropna(subset=["Female", "Male"])
    .reset_index()
)
national_wide["F_to_M"] = national_wide["Female"] / national_wide["Male"]

fig3 = px.line(
    national_wide.sort_values("Year"),
    x="Year", y="F_to_M",
    color="Education_Level",
    markers=True,
    labels={"F_to_M": "Female / Male income ratio", "Education_Level": "Education"},
    title="Are Women Catching Up? Female/Male Wage/Salary Ratio by Education Level (2004–2024)",
    hover_data={"F_to_M": ":.3f", "Year": True, "Education_Level": True},
)
fig3.add_hline(y=1.0, line_dash="dash", line_color="black",
               annotation_text="Parity = 1.00", annotation_position="top left")
fig3.update_layout(width=1030, height=650, legend_title_text="Education", template="plotly_white")
fig3.update_yaxes(range=[0.75, 1.10])
fig3.show()

latest = national_wide[national_wide["Year"] == national_wide["Year"].max()].copy()
num_above_one = (latest["F_to_M"] > 1).sum()
print(f"Education groups above parity in {int(national_wide['Year'].max())}: {num_above_one} of {latest['Education_Level'].nunique()}")

*Figure 3: National female-to-male wage/salary ratio by education level, 2004–2024. The x-axis is the calendar year and the y-axis is the ratio of women's average wage to men's average wage within the same education group, so 1.0 corresponds to perfect parity (marked by the dashed black line). Each coloured line tracks one of the six education tiers, from Primary School at the top of the bundle to Long Higher Education at the bottom. A line that slopes upward over time means women are gaining ground; a line that crosses 1.0 would mean women earn more than men in that group. The y-axis is clamped to 0.75–1.10 so that small year-on-year movements remain visible.*

**What this plot shows:** Across the full 2004–2024 window, every education line stays under the parity threshold — no group flips into women-earn-more territory. The Primary School and Gymnasium lines hover around 0.87–0.92 and sit closest to parity, while Long Higher Education forms the floor of the chart at roughly 0.73–0.75 and barely moves over twenty years. The middle tiers (Vocational, Short Higher, Medium Higher) drift upward only modestly. The shape of the bundle is essentially flat: the ordering of education groups in 2024 is almost identical to the ordering in 2004.

### Interpretation:

The figure delivers a fairly blunt verdict on the catch-up hypothesis: women are **not** meaningfully closing the wage gap inside their own education group, and the credential tier where the gap is widest is also the tier where convergence is slowest.

- **No crossings of parity.** Zero of the six education lines reach 1.0 at any point in the series. Twenty years of data, and the answer to "have women caught up?" is uniformly *no* at the national level.
- **The ladder ranking is stable.** Lower-credential groups (Primary School, Gymnasium) sit nearest parity throughout, and Long Higher Education sits furthest from parity throughout. The relative position of each tier in 2004 is essentially preserved in 2024 — there has been no reshuffling of who is closest to equal pay.
- **The top of the ladder is the flattest.** Long Higher Education shows the smallest upward slope despite women now dominating new graduate cohorts (59.1% vs. 43.7% among 25–34-year-olds in 2024). This contradicts the part of the hypothesis that predicted faster convergence where the female credential surplus is largest, and points instead to a structural ceiling at the top of the labour market — bonuses, leadership roles, and partner-track positions that the credential count alone does not unlock.
- **What this implies for the broader question.** Combined with the regional and cross-sectional evidence from Questions 1 and 2, Figure 3 suggests that simply waiting for women's educational advantage to translate into income parity is not a working strategy on the current trajectory. The descriptive picture is one of stagnation, not slow convergence — which is the empirical motivation for the structural-effect estimate produced by the ML model in the next section.